# 08 — Custom Validators

32 examples covering @register_validator, Validator base class, PassResult/FailResult,
parameterized validators, async validators, metadata access, FIX strategies,
custom REASK messages, ErrorSpan, and composing with built-in validators.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv langdetect textblob
```

In [ ]:
import os, re, asyncio, logging
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, AsyncGuard, OnFailAction
from guardrails.errors import ValidationError
from guardrails.validator_base import (
    Validator,
    register_validator,
    ValidationResult,
    PassResult,
    FailResult,
)

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

## Examples 01–02: Decorator and Class-Based Approaches

In [ ]:
# Example 01: Minimal custom validator via @register_validator decorator
@register_validator(name='no-numbers', data_type='string')
class NoNumbers(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if any(c.isdigit() for c in value):
            return FailResult(error_message='Value must not contain digits')
        return PassResult()

guard = Guard().use(NoNumbers(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Hello W0rld')  # contains digit
except ValidationError:
    print('FAIL - digits found')

outcome = guard.validate('Hello World')
print('PASS - no digits:', outcome.validation_passed)

In [ ]:
# Example 02: Full class-based Validator — starts-with check
@register_validator(name='starts-with-capital', data_type='string')
class StartsWithCapital(Validator):
    """Validates that a string begins with an uppercase letter."""

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not value or not value[0].isupper():
            return FailResult(
                error_message=f'Expected to start with capital letter, got: {value[:20]!r}'
            )
        return PassResult()

guard = Guard().use(StartsWithCapital(on_fail=OnFailAction.EXCEPTION))
guard.validate('Proper sentence.')
print('PASS - starts with capital')
try:
    guard.validate('lowercase start')
except ValidationError as e:
    print('FAIL:', str(e)[:80])

## Examples 03–06: PassResult, FailResult, and fix_value

In [ ]:
# Example 03: validate() signature — (value, metadata) -> ValidationResult
@register_validator(name='non-empty', data_type='string')
class NonEmpty(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        # metadata is always a dict — can contain sources, user_id, etc.
        if not value or not value.strip():
            return FailResult(error_message='Value cannot be empty or whitespace-only')
        return PassResult()

guard = Guard().use(NonEmpty(on_fail=OnFailAction.EXCEPTION))
guard.validate('some text')
print('PASS - non-empty string')
try:
    guard.validate('   ')
except ValidationError:
    print('FAIL - whitespace-only')

In [ ]:
# Example 04: Explicitly returning PassResult
@register_validator(name='length-ok', data_type='string')
class LengthOK(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if len(value) < 3:
            return FailResult(error_message='Too short')
        return PassResult()  # explicit PassResult

guard = Guard().use(LengthOK(on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('Hello')
print('PassResult returned, passed:', outcome.validation_passed)

In [ ]:
# Example 05: FailResult with custom error_message
@register_validator(name='no-urls', data_type='string')
class NoURLs(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        url_pattern = re.compile(r'https?://\S+')
        if url_pattern.search(value):
            return FailResult(
                error_message='Response must not contain URLs',
                fix_value=url_pattern.sub('[URL REMOVED]', value)
            )
        return PassResult()

guard = Guard().use(NoURLs(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Visit us at https://example.com for more info.')
except ValidationError as e:
    print('FAIL with message:', str(e)[:80])

In [ ]:
# Example 06: FailResult with fix_value — enables FIX action downstream
@register_validator(name='no-urls-fix', data_type='string')
class NoURLsFix(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        url_pattern = re.compile(r'https?://\S+')
        if url_pattern.search(value):
            return FailResult(
                error_message='URLs found',
                fix_value=url_pattern.sub('[LINK]', value)  # provides the fix
            )
        return PassResult()

guard = Guard().use(NoURLsFix(on_fail=OnFailAction.FIX))
outcome = guard.validate('Check https://example.com and https://docs.example.com for details.')
print('FIX applied:', outcome.validated_output)

## Examples 07–09: Parameterized and Async Validators

In [ ]:
# Example 07: Parameterized validator — keyword required in text
@register_validator(name='must-contain', data_type='string')
class MustContain(Validator):
    def __init__(self, keyword: str, **kwargs):
        self.keyword = keyword
        super().__init__(**kwargs)

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if self.keyword.lower() not in value.lower():
            return FailResult(
                error_message=f'Response must contain the keyword: {self.keyword!r}'
            )
        return PassResult()

guard = Guard().use(MustContain(keyword='disclaimer', on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate('Invest now for great returns!')
except ValidationError:
    print('FAIL - missing required keyword "disclaimer"')

outcome = guard.validate('Invest now — disclaimer: this is not financial advice.')
print('PASS - keyword present:', outcome.validation_passed)

In [ ]:
# Example 08: Multiple parameters — min and max word count
@register_validator(name='word-count', data_type='string')
class WordCount(Validator):
    def __init__(self, min_words: int = 1, max_words: int = 500, **kwargs):
        self.min_words = min_words
        self.max_words = max_words
        super().__init__(**kwargs)

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        count = len(value.split())
        if count < self.min_words:
            return FailResult(error_message=f'Too few words: {count} < {self.min_words}')
        if count > self.max_words:
            return FailResult(
                error_message=f'Too many words: {count} > {self.max_words}',
                fix_value=' '.join(value.split()[:self.max_words])  # truncate
            )
        return PassResult()

guard = Guard().use(WordCount(min_words=5, max_words=20, on_fail=OnFailAction.FIX))
outcome = guard.validate('One two three.')  # too few words
print('too few words result:', outcome.validated_output, '| passed:', outcome.validation_passed)

long_text = ' '.join([f'word{i}' for i in range(30)])  # 30 words, exceeds max of 20
outcome2 = guard.validate(long_text)
print('truncated to 20 words:', len(outcome2.validated_output.split()) if outcome2.validated_output else 0)

In [ ]:
# Example 09: Async validator — uses async def validate()
@register_validator(name='async-check', data_type='string')
class AsyncLengthCheck(Validator):
    async def validate(self, value: str, metadata: dict) -> ValidationResult:
        # simulate async work (e.g., async database lookup)
        await asyncio.sleep(0)  # zero-delay coroutine
        if len(value) < 10:
            return FailResult(error_message='Async: value too short')
        return PassResult()

async def run_async_guard():
    guard = AsyncGuard().use(AsyncLengthCheck(on_fail=OnFailAction.EXCEPTION))
    try:
        await guard.async_validate('Hi')
    except ValidationError:
        print('FAIL - async validator blocked short string')
    outcome = await guard.async_validate('A sufficiently long string passes.')
    print('PASS - async validator:', outcome.validation_passed)

asyncio.run(run_async_guard())

## Examples 10–12: AsyncGuard, Metadata, External API

In [ ]:
# Example 10: Async validator paired with AsyncGuard for LLM calls
@register_validator(name='async-non-empty', data_type='string')
class AsyncNonEmpty(Validator):
    async def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not value or not value.strip():
            return FailResult(error_message='Empty response not allowed')
        return PassResult()

async def llm_with_async_guard():
    import openai
    async_oai = openai.AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    guard = AsyncGuard().use(AsyncNonEmpty(on_fail=OnFailAction.EXCEPTION))
    outcome = await guard(
        async_oai.chat.completions.create,
        prompt='What is the capital of France?',
        model=MODEL
    )
    print('async LLM result:', outcome.validated_output)

asyncio.run(llm_with_async_guard())

In [ ]:
# Example 11: Validator reads context from metadata dict
@register_validator(name='context-aware', data_type='string')
class ContextAware(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        required_topic = metadata.get('required_topic', '')
        if required_topic and required_topic.lower() not in value.lower():
            return FailResult(
                error_message=f'Response must mention: {required_topic}'
            )
        return PassResult()

guard = Guard().use(ContextAware(on_fail=OnFailAction.EXCEPTION))
try:
    guard.validate(
        'Here is a great summary of the topic.',
        metadata={'required_topic': 'Python'}
    )
except ValidationError:
    print('FAIL - required topic Python not mentioned')

outcome = guard.validate(
    'Python is a versatile language.',
    metadata={'required_topic': 'Python'}
)
print('PASS - topic mentioned:', outcome.validation_passed)

In [ ]:
# Example 12: Custom validator that calls an external API (mocked)
import urllib.request

@register_validator(name='url-reachable', data_type='string')
class URLReachable(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not value.startswith('http'):
            return FailResult(error_message='Not a URL')
        try:
            urllib.request.urlopen(value, timeout=3)  # noqa: S310
            return PassResult()
        except Exception:
            return FailResult(error_message=f'URL not reachable: {value}')

guard = Guard().use(URLReachable(on_fail=OnFailAction.NOOP))
outcome = guard.validate('https://httpbin.org/get')
print('URL reachable:', outcome.validation_passed)

## Examples 13–18: Domain-Specific Custom Validators

In [ ]:
# Example 13: Custom validator with LRU caching for expensive calls
from functools import lru_cache

@register_validator(name='cached-check', data_type='string')
class CachedCheck(Validator):
    @lru_cache(maxsize=256)
    def _is_valid(self, value: str) -> bool:
        # Simulate expensive check (e.g., embedding lookup)
        return len(value) >= 5

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not self._is_valid(value):
            return FailResult(error_message='Too short (cached check)')
        return PassResult()

guard = Guard().use(CachedCheck(on_fail=OnFailAction.EXCEPTION))
guard.validate('Hello World')  # first call — computes
guard.validate('Hello World')  # cache hit
print('Cached validator working. Cache info:', CachedCheck()._is_valid.cache_info())

In [ ]:
# Example 14: Regex-based custom validator — wrapping a specific business rule
@register_validator(name='product-code', data_type='string')
class ProductCode(Validator):
    PATTERN = re.compile(r'^[A-Z]{2}-\d{4}-[A-Z0-9]{3}$')

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not self.PATTERN.match(value):
            return FailResult(
                error_message=f'Invalid product code format. Expected XX-NNNN-XXX, got: {value}'
            )
        return PassResult()

guard = Guard().use(ProductCode(on_fail=OnFailAction.EXCEPTION))
guard.validate('AB-1234-X5Y')
print('PASS - valid product code')
try:
    guard.validate('abc-123')
except ValidationError:
    print('FAIL - invalid product code format')

In [ ]:
# Example 15: Language detector validator — enforce English-only responses
try:
    from langdetect import detect, LangDetectException
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

@register_validator(name='english-only', data_type='string')
class EnglishOnly(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not LANGDETECT_AVAILABLE:
            return PassResult()  # skip if langdetect not installed
        try:
            lang = detect(value)
            if lang != 'en':
                return FailResult(error_message=f'Non-English language detected: {lang}')
        except Exception:
            return PassResult()  # can't detect — pass through
        return PassResult()

guard = Guard().use(EnglishOnly(on_fail=OnFailAction.EXCEPTION))
guard.validate('This is an English sentence.')
print('PASS - English text accepted')
try:
    guard.validate('Bonjour, comment allez-vous? Je suis très heureux de vous rencontrer.')
except ValidationError:
    print('FAIL - French text blocked')

In [ ]:
# Example 16: Custom PII validator — detect domain-specific IDs (employee IDs)
@register_validator(name='no-employee-id', data_type='string')
class NoEmployeeID(Validator):
    # Employee IDs follow pattern EMP-XXXXX
    PATTERN = re.compile(r'\bEMP-\d{5}\b')

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        matches = self.PATTERN.findall(value)
        if matches:
            fixed = self.PATTERN.sub('[EMPLOYEE_ID]', value)
            return FailResult(
                error_message=f'Employee IDs found: {matches}',
                fix_value=fixed
            )
        return PassResult()

guard = Guard().use(NoEmployeeID(on_fail=OnFailAction.FIX))
outcome = guard.validate('Employee EMP-12345 submitted the report and EMP-99988 approved it.')
print('FIX result:', outcome.validated_output)

In [ ]:
# Example 17: Rule-based sentiment validator
NEGATIVE_WORDS = {'bad', 'terrible', 'awful', 'horrible', 'hate', 'worst', 'disgusting'}

@register_validator(name='positive-sentiment', data_type='string')
class PositiveSentiment(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        words = set(value.lower().split())
        found_negative = words & NEGATIVE_WORDS
        if found_negative:
            return FailResult(error_message=f'Negative words detected: {found_negative}')
        return PassResult()

guard = Guard().use(PositiveSentiment(on_fail=OnFailAction.EXCEPTION))
guard.validate('This is a wonderful product that users love!')
print('PASS - positive sentiment')
try:
    guard.validate('This is terrible and the worst experience ever.')
except ValidationError:
    print('FAIL - negative words detected')

In [ ]:
# Example 18: Business rule validator — price must end in .99 or .00
@register_validator(name='price-format', data_type='string')
class PriceFormat(Validator):
    PATTERN = re.compile(r'^\$\d+\.(99|00)$')

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not self.PATTERN.match(value):
            return FailResult(error_message=f'Price must end in .99 or .00, got: {value}')
        return PassResult()

guard = Guard().use(PriceFormat(on_fail=OnFailAction.EXCEPTION))
guard.validate('$9.99')
guard.validate('$100.00')
print('PASS - valid price formats')
try:
    guard.validate('$9.95')
except ValidationError:
    print('FAIL - invalid price format')

## Examples 19–24: REASK, FIX, and Chaining

In [ ]:
# Example 19: Custom REASK message — override how LLM is instructed to fix
@register_validator(name='must-be-json', data_type='string')
class MustBeJSON(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        import json as _json
        try:
            _json.loads(value)
            return PassResult()
        except _json.JSONDecodeError:
            return FailResult(
                error_message='Response must be valid JSON',
                fix_value=None
            )

guard = Guard().use(MustBeJSON(on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Return a JSON object with keys name and age.',
    model=MODEL,
    num_reasks=2
)
print('JSON REASK result:', outcome.validated_output)

In [ ]:
# Example 20: Date format validator — strict ISO 8601
from datetime import datetime as dt

@register_validator(name='iso-date', data_type='string')
class ISODate(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        for fmt in ('%Y-%m-%d', '%Y-%m-%dT%H:%M:%S'):
            try:
                dt.strptime(value.strip(), fmt)
                return PassResult()
            except ValueError:
                continue
        return FailResult(error_message=f'Not a valid ISO date: {value}')

guard = Guard().use(ISODate(on_fail=OnFailAction.EXCEPTION))
guard.validate('2026-01-15')
print('PASS - ISO date')
try:
    guard.validate('January 15, 2026')
except ValidationError:
    print('FAIL - non-ISO date format')

In [ ]:
# Example 21: Custom FIX strategy — computed dynamically from failing value
@register_validator(name='ensure-period', data_type='string')
class EnsurePeriod(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if not value.rstrip().endswith('.'):
            return FailResult(
                error_message='Sentence must end with a period',
                fix_value=value.rstrip() + '.'  # dynamically computed fix
            )
        return PassResult()

guard = Guard().use(EnsurePeriod(on_fail=OnFailAction.FIX))
outcome = guard.validate('The sky is blue')
print('FIX adds period:', outcome.validated_output)

In [ ]:
# Example 22: Value mutation — trim whitespace and normalize case
@register_validator(name='normalize-category', data_type='string')
class NormalizeCategory(Validator):
    VALID = {'python', 'javascript', 'go', 'rust', 'java'}

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        normalized = value.strip().lower()
        if normalized not in self.VALID:
            return FailResult(
                error_message=f'{value!r} is not a valid language',
                fix_value=normalized if normalized in self.VALID else value
            )
        return PassResult()

guard = Guard().use(NormalizeCategory(on_fail=OnFailAction.FIX))
outcome = guard.validate('  Python  ')  # whitespace + wrong case
print('normalized:', outcome.validated_output)

In [ ]:
# Example 23: Chaining two custom validators
@register_validator(name='no-numbers-v2', data_type='string')
class NoNumbersV2(Validator):
    def validate(self, value, metadata):
        return FailResult('Contains digits') if any(c.isdigit() for c in value) else PassResult()

@register_validator(name='min-length-v2', data_type='string')
class MinLengthV2(Validator):
    def __init__(self, min_len: int = 10, **kwargs):
        self.min_len = min_len
        super().__init__(**kwargs)
    def validate(self, value, metadata):
        return PassResult() if len(value) >= self.min_len else FailResult(f'Min {self.min_len} chars')

guard = (
    Guard()
    .use(NoNumbersV2(on_fail=OnFailAction.EXCEPTION))
    .use(MinLengthV2(min_len=10, on_fail=OnFailAction.EXCEPTION))
)
try:
    guard.validate('Short')  # fails MinLength
except ValidationError:
    print('FAIL - too short')
outcome = guard.validate('A sufficiently long string without numbers.')
print('PASS - both validators passed:', outcome.validation_passed)

## Examples 24–32: Logging, Testing, Hub Structure, Stateful

In [ ]:
# Example 24: Validator with audit logging inside validate()
audit_log = []

@register_validator(name='audited', data_type='string')
class AuditedValidator(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        result = 'pass' if len(value) >= 5 else 'fail'
        audit_log.append({'value': value[:20], 'result': result, 'len': len(value)})
        if result == 'fail':
            return FailResult(error_message='Too short (audited)')
        return PassResult()

guard = Guard().use(AuditedValidator(on_fail=OnFailAction.NOOP))
for t in ['Hi', 'Hello', 'World!']:
    guard.validate(t)
print('Audit log:', audit_log)

In [ ]:
# Example 25: Unit testing a custom validator directly
@register_validator(name='no-all-caps', data_type='string')
class NoAllCaps(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if value == value.upper() and value.strip():
            return FailResult(error_message='All-caps text not allowed')
        return PassResult()

# Direct unit test without a Guard wrapping
validator = NoAllCaps(on_fail=OnFailAction.EXCEPTION)
assert isinstance(validator.validate('hello', {}), PassResult)
assert isinstance(validator.validate('HELLO', {}), FailResult)
assert isinstance(validator.validate('Hello World', {}), PassResult)
print('All unit tests passed for NoAllCaps validator')

In [ ]:
# Example 26: Custom REASK — override the reask prompt template
@register_validator(name='structured-output-required', data_type='string')
class StructuredOutputRequired(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        import json as _json
        try:
            _json.loads(value)
            return PassResult()
        except Exception:
            return FailResult(
                error_message='Must be valid JSON',
                fix_value=None
            )

guard = Guard().use(StructuredOutputRequired(on_fail=OnFailAction.REASK))
outcome = guard(
    oai.chat.completions.create,
    prompt='Return ONLY a JSON object: {"status": "ok", "version": 1}',
    model=MODEL,
    num_reasks=1
)
print('structured output result:', outcome.validated_output)

In [ ]:
# Example 27: ErrorSpan — mark partial failure within a string
from guardrails.validator_base import ErrorSpan

@register_validator(name='no-pii-custom', data_type='string')
class NoPIICustom(Validator):
    SSN_PATTERN = re.compile(r'\b\d{3}-\d{2}-\d{4}\b')

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        matches = list(self.SSN_PATTERN.finditer(value))
        if matches:
            # Mark each SSN span as a failure region
            error_spans = [
                ErrorSpan(start=m.start(), end=m.end(), reason='SSN detected')
                for m in matches
            ]
            return FailResult(
                error_message=f'{len(matches)} SSN(s) found',
                fix_value=self.SSN_PATTERN.sub('[SSN]', value),
                error_spans=error_spans
            )
        return PassResult()

guard = Guard().use(NoPIICustom(on_fail=OnFailAction.FIX))
outcome = guard.validate('Customer 123-45-6789 placed an order.')
print('SSN redacted:', outcome.validated_output)

In [ ]:
# Example 28: Custom validator attached to Pydantic field
from pydantic import BaseModel, Field

@register_validator(name='no-html-custom', data_type='string')
class NoHTML(Validator):
    HTML_PATTERN = re.compile(r'<[^>]+>')

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if self.HTML_PATTERN.search(value):
            return FailResult(
                error_message='HTML tags not allowed',
                fix_value=self.HTML_PATTERN.sub('', value)
            )
        return PassResult()

class Comment(BaseModel):
    author: str
    text: str

guard = Guard.for_pydantic(output_class=Comment)
guard.use(NoHTML(on_fail=OnFailAction.FIX))
outcome = guard.validate('{"author": "Alice", "text": "Great <b>product</b>!"}')
print('HTML stripped:', outcome.validated_output)

In [ ]:
# Example 29: Combining custom + built-in validators in one guard
from guardrails.hub import ValidLength

@register_validator(name='no-exclamation', data_type='string')
class NoExclamation(Validator):
    def validate(self, value: str, metadata: dict) -> ValidationResult:
        if '!' in value:
            return FailResult(error_message='Exclamation marks not professional')
        return PassResult()

guard = (
    Guard()
    .use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
    .use(NoExclamation(on_fail=OnFailAction.EXCEPTION))
)
try:
    guard.validate('This is amazing!')
except ValidationError:
    print('FAIL - exclamation mark blocked')
outcome = guard.validate('This is a professional response without exclamation marks.')
print('PASS - combined custom + built-in:', outcome.validation_passed)

In [ ]:
# Example 30: Stateful validator — tracks call count across invocations
@register_validator(name='call-counter', data_type='string')
class CallCounter(Validator):
    def __init__(self, max_calls: int = 5, **kwargs):
        self.max_calls = max_calls
        self.call_count = 0
        super().__init__(**kwargs)

    def validate(self, value: str, metadata: dict) -> ValidationResult:
        self.call_count += 1
        if self.call_count > self.max_calls:
            return FailResult(error_message=f'Rate limit exceeded: {self.call_count} calls')
        return PassResult()

counter = CallCounter(max_calls=3, on_fail=OnFailAction.EXCEPTION)
guard = Guard().use(counter)
for i in range(5):
    try:
        guard.validate(f'request {i}')
        print(f'  call {i+1}: PASS')
    except ValidationError:
        print(f'  call {i+1}: FAIL - rate limit exceeded')

In [ ]:
# Example 31: Custom validator for structured output field — price ends in .99
from pydantic import BaseModel

@register_validator(name='retail-price', data_type='float')
class RetailPrice(Validator):
    def validate(self, value, metadata: dict) -> ValidationResult:
        # Price should end in .99 or .00
        cents = round(value % 1, 2)
        if cents not in (0.99, 0.0):
            suggested = round(float(int(value)) + 0.99, 2)
            return FailResult(
                error_message=f'Price {value} must end in .99 or .00',
                fix_value=suggested
            )
        return PassResult()

class PricedItem(BaseModel):
    name: str
    price: float

guard = Guard.for_pydantic(output_class=PricedItem)
guard.use(RetailPrice(on_fail=OnFailAction.FIX))
outcome = guard.validate('{"name": "Widget", "price": 12.95}')
print('FIX retail price:', outcome.validated_output)

In [ ]:
# Example 32: Hub-compatible validator directory structure (documentation example)
hub_structure = """
my_custom_validator/
├── __init__.py
├── validator.py          # Main validator class with @register_validator
├── pyproject.toml        # Package metadata with guardrails-hub extras
├── README.md             # Usage documentation
└── tests/
    └── test_validator.py # Unit tests for the validator

# pyproject.toml essentials:
# [project]
# name = "guardrails-my-validator"
# version = "0.1.0"
# dependencies = ["guardrails-ai>=0.5.0"]
#
# [tool.guardrails-hub]
# id = "my_username/my_validator"
# validator_classname = "MyValidator"
"""
print(hub_structure)